In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
os.makedirs('../models', exist_ok=True)

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.12.1+cu130


In [2]:
DATA_PATH = '../data/'

normal_df = pd.read_csv(f'{DATA_PATH}normal3.csv', header=None, names=['raw', 'baseline', 'delta', 'current'])
slowed_df = pd.read_csv(f'{DATA_PATH}slowed3.csv', header=None, names=['raw', 'baseline', 'delta', 'current'])

print(f"Normal: {normal_df.shape}, Slowed: {slowed_df.shape}")

Normal: (1280, 4), Slowed: (1280, 4)


In [3]:
WINDOW_SIZE = 128

def create_windows(data, window_size):
    windows = []
    for i in range(len(data) - window_size + 1):
        windows.append(data[i:i+window_size])
    return np.array(windows)

X = np.vstack([
    create_windows(normal_df['current'].values, WINDOW_SIZE),
    create_windows(slowed_df['current'].values, WINDOW_SIZE)
])
y = np.hstack([
    np.zeros(len(normal_df) - WINDOW_SIZE + 1),
    np.ones(len(slowed_df) - WINDOW_SIZE + 1)
])

indices = np.random.permutation(len(X))
X, y = X[indices], y[indices]

print(f"Total samples: {X.shape}")

Total samples: (2306, 128)


In [4]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X.reshape(-1, 1)).reshape(-1, WINDOW_SIZE)
joblib.dump(scaler, '../models/scaler.joblib')

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

Train: 1613, Val: 231, Test: 462


In [5]:
class MotorFaultCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, 7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)
        self.dropout1 = nn.Dropout(0.15)
        
        self.conv2 = nn.Conv1d(32, 64, 5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)
        self.dropout2 = nn.Dropout(0.2)
        
        self.conv3 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(2)
        self.dropout3 = nn.Dropout(0.3)
        
        self.fc1 = nn.Linear(128 * 16, 64)
        self.bn_fc1 = nn.BatchNorm1d(64)
        self.dropout_fc = nn.Dropout(0.4)
        self.fc2 = nn.Linear(64, 2)
        
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout1(x)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout2(x)
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout3(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn_fc1(self.fc1(x)))
        x = self.dropout_fc(x)
        x = self.fc2(x)
        return x

model = MotorFaultCNN()
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Total parameters: 167,106


In [6]:
BATCH_SIZE = 32

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32).unsqueeze(1),
        torch.tensor(y_train, dtype=torch.long)
    ),
    batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val, dtype=torch.float32).unsqueeze(1),
        torch.tensor(y_val, dtype=torch.long)
    ),
    batch_size=BATCH_SIZE, shuffle=False
)
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test, dtype=torch.float32).unsqueeze(1),
        torch.tensor(y_test, dtype=torch.long)
    ),
    batch_size=BATCH_SIZE, shuffle=False
)

In [7]:
def train_model(model, train_loader, val_loader, epochs=50, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    
    best_val_acc = 0
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            train_total += batch_y.size(0)
            train_correct += (pred == batch_y).sum().item()
        
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                _, pred = torch.max(outputs, 1)
                val_total += batch_y.size(0)
                val_correct += (pred == batch_y).sum().item()
        
        val_acc = 100 * val_correct / val_total
        scheduler.step(val_loss)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), '../models/best_model.pth')
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= 15:
            break
    
    model.load_state_dict(torch.load('../models/best_model.pth'))
    return model, best_val_acc

model, best_val_acc = train_model(model, train_loader, val_loader)
print(f"Best validation accuracy: {best_val_acc:.2f}%")

Best validation accuracy: 100.00%


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.numpy())

accuracy = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
print(f"PyTorch Test Accuracy: {accuracy:.2f}%")

PyTorch Test Accuracy: 100.00%


In [9]:
import tensorflow as tf
import onnx2tf
import shutil
import os
import numpy as np

# Clean previous exports
for path in ["../models/model_tf", "../models/model_tf_quant"]:
    if os.path.exists(path):
        shutil.rmtree(path)

# Export to ONNX
dummy_input = torch.randn(1, 1, 128)
onnx_path = "../models/model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)
print("✅ Exported to ONNX")

# Convert to TensorFlow
onnx2tf.convert(
    input_onnx_file_path=onnx_path,
    output_folder_path="../models/model_tf",
    output_signaturedefs=True,
)
print("✅ Converted to TensorFlow")

# Verify SavedModel
saved_model = tf.saved_model.load("../models/model_tf")
concrete_func = saved_model.signatures['serving_default']
print(f"SavedModel input shape: {concrete_func.inputs[0].shape}")

# FP32 TFLite
converter = tf.lite.TFLiteConverter.from_saved_model("../models/model_tf")
tflite_model = converter.convert()
with open("../models/model_fp32.tflite", "wb") as f:
    f.write(tflite_model)
print("✅ FP32 TFLite model saved")

# FP16 TFLite
converter = tf.lite.TFLiteConverter.from_saved_model("../models/model_tf")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()
with open("../models/model_fp16.tflite", "wb") as f:
    f.write(tflite_model)
print("✅ FP16 TFLite model saved")

2026-07-12 08:00:12.987964: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-12 08:00:12.990730: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-12 08:00:12.994990: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783823413.002317  189523 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783823413.004650  189523 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been regist

[torch.onnx] Obtain model graph for `MotorFaultCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MotorFaultCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnx/version_converter.py", line 39, in convert_vers

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ Exported to ONNX

Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Concat     │ 1              │ 1                │
│ Constant   │ 11             │ 11               │
│ Conv       │ 3              │ 3                │
│ Gemm       │ 2              │ 2                │
│ MaxPool    │ 3              │ 3                │
│ Relu       │ 4              │ 4                │
│ Reshape    │ 1              │ 1                │
│ Shape      │ 1              │ 1                │
│ Model Size │ 666.2KiB       │ 653.0KiB         │
└────────────┴────────

2026-07-12 08:00:15.977570: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


...

W0000 00:00:1783823417.752229  189523 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1783823417.752243  189523 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-07-12 08:00:17.752354: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp4oda0wwd
2026-07-12 08:00:17.752824: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-07-12 08:00:17.752831: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp4oda0wwd
2026-07-12 08:00:17.754643: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-07-12 08:00:17.758775: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp4oda0wwd
2026-07-12 08:00:17.761895: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 9542 microseconds.


Float16 tflite output complete!
✅ Converted to TensorFlow
SavedModel input shape: (None, 128, 1)


W0000 00:00:1783823418.205106  189523 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1783823418.205125  189523 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-07-12 08:00:18.205274: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: ../models/model_tf
2026-07-12 08:00:18.205648: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-07-12 08:00:18.205654: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: ../models/model_tf
2026-07-12 08:00:18.207135: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-07-12 08:00:18.210978: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: ../models/model_tf
2026-07-12 08:00:18.213846: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 8576 microseconds.


✅ FP32 TFLite model saved
✅ FP16 TFLite model saved


W0000 00:00:1783823418.606263  189523 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1783823418.606281  189523 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-07-12 08:00:18.606432: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: ../models/model_tf
2026-07-12 08:00:18.606924: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-07-12 08:00:18.606932: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: ../models/model_tf
2026-07-12 08:00:18.608702: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-07-12 08:00:18.612700: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: ../models/model_tf
2026-07-12 08:00:18.616096: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 9668 microseconds.


In [10]:
def verify_tflite(path):
    interpreter = tf.lite.Interpreter(model_path=path)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()
    out = interpreter.get_output_details()
    print(f"  Input: {inp[0]['shape']}, Output: {out[0]['shape']}")
    return interpreter

print("FP32 Model:")
verify_tflite("../models/model_fp32.tflite")
print("\nFP16 Model:")
verify_tflite("../models/model_fp16.tflite")

FP32 Model:
  Input: [  1 128   1], Output: [1 2]

FP16 Model:
  Input: [  1 128   1], Output: [1 2]


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [11]:
def verify_tflite(path):
    interpreter = tf.lite.Interpreter(model_path=path)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()
    out = interpreter.get_output_details()
    print(f"  Input: {inp[0]['shape']}, Output: {out[0]['shape']}")
    return interpreter

print("FP32 Model:")
verify_tflite("../models/model_fp32.tflite")

print("\nFP16 Model:")
verify_tflite("../models/model_fp16.tflite")

FP32 Model:
  Input: [  1 128   1], Output: [1 2]

FP16 Model:
  Input: [  1 128   1], Output: [1 2]


In [12]:
def compare_pytorch_tflite(sample_window, tflite_path):
    scaler = joblib.load('../models/scaler.joblib')
    window_scaled = scaler.transform(sample_window.reshape(-1, 1))
    
    # PyTorch
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MotorFaultCNN()
    model.load_state_dict(torch.load('../models/best_model.pth', map_location=device))
    model.to(device)
    model.eval()
    input_torch = torch.tensor(window_scaled.reshape(1, 1, 128), dtype=torch.float32).to(device)
    with torch.no_grad():
        torch_out = model(input_torch).cpu().numpy()[0]
        torch_pred = np.argmax(torch_out)
        torch_conf = np.exp(torch_out[torch_pred]) / np.sum(np.exp(torch_out))
    
    # TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()
    out = interpreter.get_output_details()
    input_tflite = window_scaled.reshape(1, 128, 1).astype(np.float32)
    interpreter.set_tensor(inp[0]['index'], input_tflite)
    interpreter.invoke()
    tflite_out = interpreter.get_tensor(out[0]['index'])[0]
    tflite_pred = np.argmax(tflite_out)
    tflite_conf = np.exp(tflite_out[tflite_pred]) / np.sum(np.exp(tflite_out))
    
    return {
        'torch': {'output': torch_out, 'pred': torch_pred, 'conf': float(torch_conf)},
        'tflite': {'output': tflite_out, 'pred': tflite_pred, 'conf': float(tflite_conf)}
    }

normal_sample = normal_df['current'].values[:128]
fault_sample = slowed_df['current'].values[:128]

print("FP32 Comparison:")
for name, sample in [("Normal", normal_sample), ("Fault", fault_sample)]:
    res = compare_pytorch_tflite(sample, "../models/model_fp32.tflite")
    print(f"  {name}: PyTorch pred={res['torch']['pred']}, TFLite pred={res['tflite']['pred']}")

FP32 Comparison:
  Normal: PyTorch pred=0, TFLite pred=0
  Fault: PyTorch pred=1, TFLite pred=1


In [13]:
def evaluate_tflite_correct(tflite_path, X_test, y_test):
    scaler = joblib.load('../models/scaler.joblib')
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()
    out = interpreter.get_output_details()
    
    # Ensure input shape is (1, 128, 1)
    assert inp[0]['shape'][1] == 128, f"Expected time dimension 128, got {inp[0]['shape'][1]}"
    assert inp[0]['shape'][2] == 1, f"Expected channels 1, got {inp[0]['shape'][2]}"
    
    predictions = []
    for window in X_test:
        # Scale and reshape
        window_scaled = scaler.transform(window.reshape(-1, 1))
        input_data = window_scaled.reshape(1, 128, 1).astype(np.float32)
        interpreter.set_tensor(inp[0]['index'], input_data)
        interpreter.invoke()
        output = interpreter.get_tensor(out[0]['index'])[0]
        predictions.append(np.argmax(output))
    
    acc = 100 * np.mean(np.array(predictions) == y_test)
    return acc

print("="*60)
print("CORRECTED TFLITE EVALUATION")
print("="*60)

acc_correct_fp32 = evaluate_tflite_correct("../models/model_fp32.tflite", X_test, y_test)
print(f"FP32 Corrected Accuracy: {acc_correct_fp32:.2f}%")

acc_correct_fp16 = evaluate_tflite_correct("../models/model_fp16.tflite", X_test, y_test)
print(f"FP16 Corrected Accuracy: {acc_correct_fp16:.2f}%")

CORRECTED TFLITE EVALUATION
FP32 Corrected Accuracy: 50.00%
FP16 Corrected Accuracy: 50.00%
